# Digital Asset Regulatory Transfer Analysis

This notebook analyzes the regulated virtual asset transfer use case for FPG. It intentionally separates regulatory evidence, schema availability, synthetic transfer evaluation, and candidate handoff.

## tl;dr

Run all cells to refresh the observed counts. No final runtime policy or legal judgment is asserted here.

## Context & Methods

The fixed sequence is regulatory source structure -> legal controls -> required fields -> transaction schema mapping -> schema gap -> synthetic regulatory transfer dataset -> policy experiment -> candidate policy.

In [ ]:
# ruff: noqa: E501, E702, I001

from pathlib import Path
import itertools
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().resolve()
if ROOT.name != 'ADP-DA':
    ROOT = next(p for p in [ROOT, *ROOT.parents] if p.name == 'ADP-DA')
REG_PATH = ROOT / '03_digital_asset/data/raw/regulatory/crypto_regulation_raw.csv'
GAP_PATH = ROOT / '03_digital_asset/data/processed/regulatory_schema_gap.csv'
SYN_PATH = ROOT / '03_digital_asset/data/processed/synthetic_regulatory_transfer_v1.csv'
reg = pd.read_csv(REG_PATH)
gap = pd.read_csv(GAP_PATH)
tx = pd.read_csv(SYN_PATH)
reg.head()

## Data

Inputs are local artifacts generated by the Digital Asset pipeline builder. Public datasets are not downloaded into this repository.

In [ ]:
print({'regulatory_rows': len(reg), 'schema_gap_fields': len(gap), 'synthetic_transfers': len(tx)})
display(reg[['source','jurisdiction','obligation_id','control_type','required_field','fpg_scope']].head(10))

## Results

### 1. Korea Regulatory Control Frequency

In [ ]:
korea_freq = reg[reg['jurisdiction']=='KR']['control_type'].value_counts().rename_axis('control_type').reset_index(name='count')
display(korea_freq)
korea_freq.plot.bar(x='control_type', y='count', legend=False, color='#3f7f93', title='Korea Control Frequency')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()

### 2. FATF Control Frequency

In [ ]:
fatf_freq = reg[reg['jurisdiction']=='GLOBAL']['control_type'].value_counts().rename_axis('control_type').reset_index(name='count')
display(fatf_freq)
fatf_freq.plot.bar(x='control_type', y='count', legend=False, color='#446fb3', title='FATF Control Frequency')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()

### 3. Korea x FATF Control Cross-tab

In [ ]:
cross_tab = pd.crosstab(reg['control_type'], reg['jurisdiction'])
display(cross_tab)

### 4. Common Control Intersection

In [ ]:
kr_controls = set(reg.loc[reg['jurisdiction']=='KR','control_type'])
fatf_controls = set(reg.loc[reg['jurisdiction']=='GLOBAL','control_type'])
common_controls = sorted(kr_controls & fatf_controls)
print(common_controls)

### 5. Jaccard Similarity

In [ ]:
jaccard = len(kr_controls & fatf_controls) / len(kr_controls | fatf_controls)
print({'jaccard_similarity': round(jaccard, 4), 'intersection': len(kr_controls & fatf_controls), 'union': len(kr_controls | fatf_controls)})

### 6. Control Co-occurrence

In [ ]:
pairs = []
for regulation, group in reg.groupby('regulation_name'):
    for a, b in itertools.combinations(sorted(set(group['control_type'])), 2):
        pairs.append({'pair': f'{a} | {b}', 'regulation_name': regulation})
cooccurrence = pd.DataFrame(pairs).groupby('pair').size().reset_index(name='count').sort_values('count', ascending=False) if pairs else pd.DataFrame(columns=['pair','count'])
display(cooccurrence.head(15))

### 7. Required Field Frequency

In [ ]:
field_freq = reg[reg['required_field']!='source_note']['required_field'].value_counts().rename_axis('required_field').reset_index(name='count')
display(field_freq)
field_freq.plot.bar(x='required_field', y='count', legend=False, color='#4f8f65', title='Required Field Frequency')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()

### 8. FPG Scope Ratio

In [ ]:
scope_ratio = reg['fpg_scope'].value_counts(normalize=True).mul(100).round(2).rename_axis('fpg_scope').reset_index(name='pct')
display(scope_ratio)

### 9. CURRENT / FUTURE Regulation Split

In [ ]:
status_split = reg['effective_status'].value_counts().rename_axis('effective_status').reset_index(name='count')
display(status_split)

### 10. Regulation -> Control -> Required Field

In [ ]:
mapping = reg[['regulation_name','article','obligation_id','control_type','required_field','fpg_scope']].sort_values(['regulation_name','control_type','required_field'])
display(mapping)

## Schema Gap

Public blockchain data is compared with legal required fields. Identity/KYC/counterparty VASP results remain outside public on-chain data.

In [ ]:
display(gap[['field','availability_class','gap_status','legal_basis']])

## Policy Experiment

The experiment checks single-condition and compound-condition behavior on synthetic transfers derived from documented public dataset shape/proportions.

In [ ]:
decision_counts = tx['policy_decision'].value_counts().rename_axis('policy_decision').reset_index(name='count')
display(decision_counts)
checks = ['asset_match','amount_within_limit','destination_match','period_valid','kyc_verified','counterparty_registered','execution_trace_present']
failure_rates = (1 - tx[checks].mean()).mul(100).round(2).rename_axis('check').reset_index(name='failure_pct')
display(failure_rates)

## Takeaways

The repeatable evidence is structural: public on-chain data can support addresses, amount, asset, tx hash, execution status, and timestamp, but cannot by itself enforce regulated transfer conditions requiring identity, KYC, approved policy, and counterparty VASP status.